# Generating games with GAMUT

[GAMUT](http://gamut.stanford.edu/) is a suite of parameterised game generators covering a wide range of game families studied in the game theory literature. Written in Java, GAMUT can generate instances of 35 game classes, including random games, coordination games, covariant games, voting games, and many more.

PyGambit's `generate_gamut` function calls GAMUT as an external subprocess and returns the resulting game as a `Game` object, ready for analysis with PyGambit's full suite of tools. Before running this tutorial, you will need Java and `gamut.jar` installed; see the [catalog documentation](../../catalog.html#catalog-gamut) for full installation instructions.

> **Note:** The cell outputs in this notebook were generated locally. To reproduce them, update the `gamut_jar` argument in each cell to the path of your local `gamut.jar`.

This tutorial covers:
- Generating classic two-player games with no additional parameters
- Generating parameterised random normal-form games
- Exploring how the covariance parameter in `CovariantGame` affects equilibrium structure
- Generating multi-player games
- Controlling payoff normalisation and obtaining integer payoffs


In [1]:
import pygambit as gbt

## Discovering available game classes

`gbt.catalog.gamut_games()` returns a DataFrame listing all 35 GAMUT game classes, their descriptions, and whether they support more than two players. Use this as a quick reference when choosing a game to generate:


In [2]:
gbt.catalog.gamut_games()


,Class,Description,Players
0,ArmsRace,Arms race with cost and demand functions chose...,n
1,BattleOfTheSexes,Coordination game where players prefer to meet...,2
2,BertrandOligopoly,Bertrand oligopoly with arbitrary cost and dem...,n
3,BidirectionalLEG,Bidirectional local-effect game on a specified...,n
4,Chicken,Classic 2x2 Chicken game.,2
5,CollaborationGame,Game where all players choosing the same actio...,n
6,CongestionGame,Congestion game where payoffs depend on how ma...,n
7,CoordinationGame,Pure coordination game rewarding matching acti...,n
8,CournotDuopoly,Cournot duopoly with arbitrary cost and invers...,2
9,CovariantGame,Game with payoffs whose cross-player correlati...,n


## Classic two-player games

Many of GAMUT's game classes correspond directly to well-known games from the game theory literature and require no additional parameters to generate. Here we generate Battle of the Sexes, a canonical 2×2 coordination game.

In Battle of the Sexes, two players must independently decide whether to go to the Opera or a Football match. Both prefer to attend the same event, but player 1 prefers the Opera while player 2 prefers Football. The game has two pure-strategy Nash equilibria — both go to the Opera, or both go to Football — and one mixed-strategy equilibrium.


In [3]:
g_bos = gbt.catalog.generate_gamut(
    "BattleOfTheSexes",
    params={
        "int_payoffs": True,
        "int_mult": 1,
        "normalize": True,
        "min_payoff": 0,
        "max_payoff": 3
    },
    gamut_jar="~/Downloads/gamut.jar",
)
g_bos.title = "Battle of the Sexes"
for player in g_bos.players:
    for strategy, label in zip(player.strategies, ["Opera", "Football"], strict=True):
        strategy.label = label
g_bos


Game(title='Battle of the Sexes')

Now let's compute all Nash equilibria using the linear complementarity method, which is well-suited to two-player games:


In [4]:
result_bos = gbt.nash.lcp_solve(g_bos)
len(result_bos.equilibria)


3

As expected, the solver finds three equilibria: two pure-strategy equilibria (both play Opera, or both play Football) and one mixed-strategy equilibrium in which each player randomises between the two options:


In [5]:
result_bos.equilibria[0]

[[Rational(1, 1), Rational(0, 1)], [Rational(1, 1), Rational(0, 1)]]

In [6]:
result_bos.equilibria[1]

[[Rational(1, 2), Rational(1, 2)], [Rational(1, 2), Rational(1, 2)]]

In [7]:
result_bos.equilibria[2]

[[Rational(0, 1), Rational(1, 1)], [Rational(0, 1), Rational(1, 1)]]

## Payoff normalisation and integer payoffs

By default, GAMUT draws payoffs uniformly from the range [-100, 100]. These global parameters let you rescale and discretise payoffs:

| Parameter | Type | Effect |
|---|---|---|
| `normalize` | boolean flag | Enable rescaling to [`min_payoff`, `max_payoff`] |
| `min_payoff` | number | Lower bound after normalisation |
| `max_payoff` | number | Upper bound after normalisation |
| `int_payoffs` | boolean flag | Round all payoffs to integers |
| `int_mult` | boolean flag | Multiplier used before rounding when converting from double to integer payoffs |

See the [GAMUT documentation](http://gamut.stanford.edu/) for more info on the available Global parameters.


## Parameterised random normal-form games

Most GAMUT game classes accept parameters to control the number of players, the number of actions, and the structure of payoffs. These are passed as a dictionary to the `params` argument; each key maps to a GAMUT command-line flag.

`RandomGame` is the most general generator: payoffs are drawn independently and uniformly at random from a fixed range. It serves as a useful baseline for testing algorithms. The `players` parameter controls the number of players, and `actions` controls the number of actions per player. When `actions` is a list, each element gives the action count for the corresponding player, allowing asymmetric games:


In [8]:
g_rand = gbt.catalog.generate_gamut(
    "RandomGame",
    params={"players": 2, "actions": [3, 3]},
    gamut_jar="~/Downloads/gamut.jar",
)
g_rand.title = "Random Game (2 players, 3x4)"
g_rand


Game(title='Random Game (2 players, 3x4)')

In [9]:
gbt.nash.lcp_solve(g_rand).equilibria[0]


[[Rational(1, 1), Rational(0, 1), Rational(0, 1)], [Rational(0, 1), Rational(0, 1), Rational(1, 1)]]

## Covariant games

`CovariantGame` generates two-player games in which the degree of alignment between players' interests is controlled by a covariance parameter `r`.

- When `r = 1` the game is a common-payoff game: both players receive identical payoffs.
- When `r = 0` payoffs are independent — equivalent to `RandomGame`.
- As `r` approaches `-1` (the minimum for a two-player game) the game approaches a zero-sum game.

Let's compare an equilibrium under high positive covariance (nearly a coordination game) with one under negative covariance (a more adversarial setting):


In [10]:
g_cov_pos = gbt.catalog.generate_gamut(
    "CovariantGame",
    params={"players": 2, "actions": [3, 3], "r": 0.8},
    gamut_jar="~/Downloads/gamut.jar",
)
g_cov_pos.title = "Covariant Game (r=0.8)"
eqm_pos = gbt.nash.lcp_solve(g_cov_pos).equilibria[0]
eqm_pos


[[Rational(1, 1), Rational(0, 1), Rational(0, 1)], [Rational(1, 1), Rational(0, 1), Rational(0, 1)]]

In [11]:
g_cov_neg = gbt.catalog.generate_gamut(
    "CovariantGame",
    params={"players": 2, "actions": [3, 3], "r": -0.5},
    gamut_jar="~/Downloads/gamut.jar",
)
g_cov_neg.title = "Covariant Game (r=-0.5)"
eqm_neg = gbt.nash.lcp_solve(g_cov_neg).equilibria[0]
eqm_neg


[[Rational(1, 1), Rational(0, 1), Rational(0, 1)], [Rational(0, 1), Rational(0, 1), Rational(1, 1)]]

With high positive covariance (`r = 0.8`) the game is close to a coordination game, so an equilibrium in pure or near-pure strategies is typical. With negative covariance (`r = -0.5`) the game is more adversarial, making genuinely mixed equilibria more likely.


## Multi-player games

GAMUT includes several n-player game families. Here we use `MajorityVoting`, a model of a committee in which each player votes for one of several candidates. The candidate who receives the most votes wins, and each player has a privately known preferred candidate.

Voting games naturally lend themselves to pure-strategy analysis — each player simply votes for their preferred candidate. `gbt.nash.enumpure_solve` searches exhaustively for pure-strategy Nash equilibria and works for games with any number of players:


In [12]:
g_mv = gbt.catalog.generate_gamut(
    "MajorityVoting",
    params={
        "int_payoffs": True,
        "int_mult": 1,
        "players": 3,
        "actions": 3
    },
    gamut_jar="~/Downloads/gamut.jar",
)
g_mv.title = "Majority Voting (3 players, 3 candidates)"
g_mv


Game(title='Majority Voting (3 players, 3 candidates)')

In [13]:
result_mv = gbt.nash.enumpure_solve(g_mv)
result_mv.equilibria[0]


[[Rational(1, 1), Rational(0, 1), Rational(0, 1)], [Rational(1, 1), Rational(0, 1), Rational(0, 1)], [Rational(1, 1), Rational(0, 1), Rational(0, 1)]]